In [ ]:
# TODO: move to import
def Cos(x, y, s, n=10):
    # 1 - x**2/2 + x**4/(4*3*2) - ...
    terms = [1]
    for i in range(2, n*2, 2):
        terms.append(x**(i)/math.factorial(i))
        if i % 4 == 2:
            terms[-1] = -1*terms[-1]

    s.add(Abs(y - sum(terms)) < 10**(-n))

def Sin(x, y, s, n=10):
    # x - x**3/(3*2) + x**5/(5*4*3*2) - ...
    terms = []
    for i in range(0, n*2, 2):
        terms.append(x**(i+1)/math.factorial(i+1))
        if i % 4 == 2:
            terms[-1] = -1*terms[-1]

    s.add(Abs(y - sum(terms)) < 10**(-n))

# Projectile Motion
Projectile motion is defined as the motion of an object that is projected (thrown or launched) into air with only the force of gravity acting upon it once launched.  
Typically we will ignore air resistance and other effects (such as spinning) when studying projectile motion.

# Splitting 2D motion into $x$ and $y$ components
We can take 2-dimensional projectile motion and split it into two parts:
the horizontal motion, and the vertical motion.  
In our case, since projectile motion only has the force of gravity acting on the projectile, we can say two things:

1. The acceleration in the $y$ direction is equal to the acceleration due to gravity.
2. The acceleration in the $x$ direction is equal to 0

Let's initialize our solver and add those two constraints first.

In [ ]:
solver = Solver()

g = Real("g")
a_x, a_y = Reals("a_x a_y")

solver.add(g == 9.81) # 9.81 m/s^2
solver.add(a_x == 0)  # acceleration in the x direction # REPLACE THIS LINE
solver.add(a_y == -g) # acceleration in the y direction

# Horizontal Motion
We can use the kinematic equations to model the horizontal motion of a projectile.
$$
v_t = v_0 + a t \\
x_t = x_0 + v_0 t + \frac{1}{2} a t^2 \\
x_t = x_0 + \frac{v_t + v_0}{2} t \\
v_t^2 = v_0^2 + 2 a (x_t - x_0)
$$
Since $a_x = 0$, we can simplify 
$$
v_{tx} = v_{0x} \\
x_{tx} = x_{0x} + v_{tx} t \\
$$
Let's add these constraints to our solver.

In [ ]:
v_0x, v_tx, x_0, x_t, t = Reals("v_0x v_tx x_0 x_t t")

solver.add(x_t == x_0 + v_tx*t) # REPLACE THIS LINE

# Vertical Motion
For vertical motion we have the following:
$$
v_{ty} = v_{0y} - g t \\
y_t = y_0 + v_{0y} t - \frac{1}{2} g t^2 \\
y_t = y_0 + \frac{v_{ty} + v_{0y}}{2} t \\
v_{ty}^2 = v_{0y}^2 - 2 g (y_t - y_0)
$$
Let's add these constraints to our solver.

In [ ]:
v_0y, v_ty, y_0, y_t = Reals("v_0y v_ty y_0 y_t")

solver.add(v_ty == v_0y - g*t) # REPLACE THIS LINE
solver.add(y_t == y_0 + v_0y*t - g*t**2/RealVal(2)) # REPLACE THIS LINE
solver.add(y_t == y_0 + (v_ty + v_0y)/RealVal(2)*t) # REPLACE THIS LINE
solver.add(v_ty**2 == v_0y**2 - 2*g*(y_t - y_0)) # REPLACE THIS LINE

Now we can test our solver by giving it some initial constraints.  
Consider the following problem:  
A cannonball is launched from a 20 meter high building at an angle of 45 degrees, with an initial velocity of 20 meters per second.  
How long does it take for the cannonball to land?

First we need to split the initial velocity into $x$ and $y$ components.  
We can do that using trigonometry, and in fact, we will use Z3 to do that for us using the following relations:
$$
v_x = v \cos(\theta) \\
v_y = v \sin(\theta)
$$

In [ ]:
th_0, cos_th0, sin_th0 = Reals("th_0 cos_th0 sin_th0")
th_t, cos_tht, sin_tht = Reals("th_t cos_tht sin_tht")
v_0, v_t = Reals("v_0 v_t")

Cos(th_0, cos_th0, solver)
Sin(th_0, sin_th0, solver)
Cos(th_t, cos_tht, solver)
Sin(th_t, sin_tht, solver)

solver.add(v_0x == v_0*cos_th0)
solver.add(v_0y == v_0*sin_th0)
solver.add(v_tx == v_t*cos_th0) # REPLACE THIS LINE
solver.add(v_ty == v_t*sin_th0) # REPLACE THIS LINE

Now we can get back to solving our original problem.  
From the problem, we can gather:
$$
v_0 = 20 \\
\theta_0 = 45 \\
y_0 = 20 \\
x_0 = 0 \\
$$
And we want to know the time $t$ when $y_t = 0$.

In [ ]:
solver.add(v_0 == 20)
solver.add(th_0 == math.radians(45)) # need to convert degrees to radians
solver.add(y_0 == 20)
solver.add(x_0 == 0)

solver.add(y_t == 0)

print(solver.check())
print(solver.model())
print(f"t (time) = {solver.model()[t]}") # should be ~3.9 seconds

To make things easier (particularly for graphing) we can wrap a class around the constraints we added to the solver.

In [ ]:
class ProjectileMotion:
    def __init__(self, solver = Solver()):
        self.solver = solver
        self.trig_param = 10

        self.g, self.t = Reals("g t")

        self.y_0, self.x_0 = Reals("y_0 x_0") # initial y position, initial x position
        self.y, self.x = Reals("y x") # y position, x position

        self.v_0, self.v_y0, self.v_x0 = Reals("v_0 v_y0 v_x0") # initial velocities
        self.th_0, self.cos_th0, self.sin_th0 = Reals("th_0 cos_th0 sin_th0") # initial angle in radians

        self.v, self.v_y, self.v_x = Reals("v v_y v_x") # velocities
        self.th, self.cos_th, self.sin_th = Reals("th cos_th sin_th") # angle in radians

        self.solver.add(self.g == 9.81)

        self.solver.add(self.x == self.x_0 + self.v_x*self.t)
        self.solver.add(self.y == self.y_0 + (self.v_y0 + self.v_y)*self.t/RealVal(2))
        self.solver.add(self.y == self.y_0 + self.v_y0*self.t - (self.g*self.t**2)/RealVal(2))

        Cos(self.th_0, self.cos_th0, self.solver, n = self.trig_param)
        Sin(self.th_0, self.sin_th0, self.solver, n = self.trig_param)
        self.solver.add(self.v_x0 == self.v_0 * self.cos_th0)
        self.solver.add(self.v_y0 == self.v_0 * self.sin_th0)

        Cos(self.th, self.cos_th, self.solver, n = self.trig_param)
        Sin(self.th, self.sin_th, self.solver, n = self.trig_param)
        self.solver.add(self.v_x == self.v * self.cos_th)
        self.solver.add(self.v_y == self.v * self.sin_th)
        self.solver.add(self.v_x == self.v_x0)
        self.solver.add(self.v_y == self.v_y0 - self.g*self.t)

    def get_value_from_model(self, variable):
        if self.solver.check() == unsat:
            return None
        else:
            return self.solver.model()[variable].py_value()

We can then use this class to easily solve any projectile motion problem we may come across.  

For example:  
A soccer ball is kicked into the air with an initial velocity of 35 meters per second (wow!) at 30 degrees above the horizontal.  
What is the maximum height the soccer ball reaches? At what time? How far did it travel horizontally at that point?

Note that a projectile reaches its highest point at the exact time that its vertical velocity $v_y = 0$.

In [ ]:
pm = ProjectileMotion()

# initial values
pm.solver.add(pm.x_0 == 0) # REPLACE THIS LINE
pm.solver.add(pm.y_0 == 0) # REPLACE THIS LINE
pm.solver.add(pm.v_0 == 35) # REPLACE THIS LINE
pm.solver.add(pm.th_0 == math.radians(30)) # REPLACE THIS LINE
pm.solver.add(pm.v_y0 >= 0) # REPLACE THIS LINE
pm.solver.add(pm.v_x0 >= 0) # REPLACE THIS LINE

# at the apex
pm.solver.add(pm.v_y == 0) # REPLACE THIS LINE

print(pm.solver.check())
print(pm.solver.model())

print("\nat the apex:")
print(f"y (height) = {pm.get_value_from_model(pm.y)}")                  # should get ~15.6 m
print(f"t (time) = {pm.get_value_from_model(pm.t)}")                    # should get ~1.8 s
print(f"x (horizontal displacement) = {pm.get_value_from_model(pm.x)}") # should get ~54.1 m

draw_projectile(pm)